# Comparative Analysis of Custom CNN and Transfer Learning Models with Explainable AI

Dataset: [Fresh and Stale Classification](https://www.kaggle.com/datasets/swoyam2609/fresh-and-stale-classification)

This notebook trains and compares:

- Custom CNN
- MobileNetV2 transfer learning model
- ResNet50 transfer learning model

It saves:

- All figures as PDF files in `/kaggle/working/outputs/figures`
- All tables as CSV files in `/kaggle/working/outputs/tables`

In [2]:
import importlib.util
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

try:
    import shap
except ImportError:
    shap = None
    print("SHAP is not installed. The notebook will skip SHAP plots instead of failing.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 130

PROJECT_ROOT = Path.cwd()
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
DATASET_SLUG = "fresh-and-stale-classification"
DATASET_URL = "https://www.kaggle.com/datasets/swoyam2609/fresh-and-stale-classification"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

SPLIT_ALIASES = {
    "train": {"train", "training"},
    "val": {"val", "valid", "validation", "dev"},
    "test": {"test", "testing"},
}
ALL_SPLIT_ALIASES = set().union(*SPLIT_ALIASES.values())


def has_images(path):
    return path.exists() and any(
        p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob("*")
    )


def find_dataset_root():
    env_root = os.environ.get("DATA_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser()
        if has_images(candidate):
            return candidate
        raise FileNotFoundError(f"DATA_ROOT is set to {candidate}, but no images were found there.")

    kaggle_candidate = KAGGLE_INPUT_ROOT / DATASET_SLUG
    if has_images(kaggle_candidate):
        return kaggle_candidate

    if KAGGLE_INPUT_ROOT.exists():
        input_dirs = [p for p in KAGGLE_INPUT_ROOT.glob("*") if p.is_dir() and has_images(p)]
        if len(input_dirs) == 1:
            return input_dirs[0]

    local_candidates = [PROJECT_ROOT / "Data", PROJECT_ROOT / "data", PROJECT_ROOT]
    for candidate in local_candidates:
        if has_images(candidate):
            return candidate

    raise FileNotFoundError(
        "No image files were found. Put the dataset inside a Data folder next to this notebook, "
        "attach the Kaggle dataset, or set the DATA_ROOT environment variable."
    )


DATA_ROOT = find_dataset_root()
OUTPUT_DIR = (KAGGLE_WORKING_ROOT / "outputs") if KAGGLE_WORKING_ROOT.exists() else (PROJECT_ROOT / "outputs")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
for directory in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
FINE_TUNE_EPOCHS = 8
FINE_TUNE_LAST_LAYERS = 40
TRAIN_FRACTION = 0.80
VAL_FRACTION = 0.10
TEST_FRACTION = 0.10

SHAP_IMAGE_COUNT = 1
SHAP_MAX_EVALS = 300
SHAP_BATCH_SIZE = 16

print(f"TensorFlow: {tf.__version__}")
print(f"Dataset root: {DATA_ROOT}")
print(f"Outputs: {OUTPUT_DIR}")
print(
    f"Split target: train={TRAIN_FRACTION:.0%}, "
    f"validation={VAL_FRACTION:.0%}, test={TEST_FRACTION:.0%}"
)


2026-06-07 13:01:19.654356: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780837279.843134      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780837279.903446      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780837280.362085      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780837280.362127      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780837280.362130      58 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
Dataset root: /kaggle/input/datasets
Outputs: /kaggle/working/outputs
Split target: train=80%, validation=10%, test=10%


## Utility Functions

In [3]:
def save_pdf(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    plt.close(fig)
    return path


def save_csv(df, filename, index=False):
    path = TABLE_DIR / filename
    df.to_csv(path, index=index)
    print(f"Saved table: {path}")
    return path


def safe_stratify(labels):
    counts = pd.Series(labels).value_counts()
    if len(counts) < 2 or counts.min() < 2:
        return None
    return labels


def stratified_split(df, test_size, random_state=SEED):
    stratify = safe_stratify(df["label"])
    return train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify,
    )


def infer_split(path):
    lower_parts = [part.lower() for part in Path(path).parts]
    for split_name, aliases in SPLIT_ALIASES.items():
        if any(part in aliases for part in lower_parts):
            return split_name
    return "unsplit"


def infer_label(path, root):
    relative_parts = Path(path).relative_to(root).parts[:-1]
    label_candidates = [part for part in relative_parts if part.lower() not in ALL_SPLIT_ALIASES]
    if label_candidates:
        return label_candidates[-1]
    return Path(path).parent.name


def load_image_tensor(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32)


def load_image_for_tf(path, label):
    return load_image_tensor(path), label


def load_weighted_image_for_tf(path, label, sample_weight):
    return load_image_tensor(path), label, sample_weight


def make_dataset(df, shuffle=False, class_weights=None):
    paths = df["path"].astype(str).values
    labels = df["label_index"].astype(np.int32).values

    if class_weights is None:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    else:
        weights = np.array([class_weights[int(label)] for label in labels], dtype=np.float32)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels, weights))

    if shuffle:
        ds = ds.shuffle(buffer_size=max(len(df), 1), seed=SEED, reshuffle_each_iteration=True)

    if class_weights is None:
        ds = ds.map(load_image_for_tf, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(load_weighted_image_for_tf, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


def read_resized_image(path):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(img, dtype=np.float32)


def make_slug(text):
    return text.replace(" ", "_").replace("/", "_").lower()


## Dataset Discovery and Description

In [4]:
image_paths = sorted(
    p for p in DATA_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

if not image_paths:
    raise FileNotFoundError(f"No image files found under {DATA_ROOT}")

records = []
for path in image_paths:
    records.append(
        {
            "path": str(path),
            "label": infer_label(path, DATA_ROOT),
            "split": infer_split(path),
            "extension": path.suffix.lower(),
        }
    )

df = pd.DataFrame(records)
class_names = sorted(df["label"].unique())
label_to_index = {label: idx for idx, label in enumerate(class_names)}
df["label_index"] = df["label"].map(label_to_index)
num_classes = len(class_names)

if num_classes < 2:
    raise ValueError(f"At least two classes are required. Found: {class_names}")

explicit_splits = set(df["split"]) - {"unsplit"}
if explicit_splits:
    train_df = df[df["split"] == "train"].copy()
    val_df = df[df["split"] == "val"].copy()
    test_df = df[df["split"] == "test"].copy()
    unsplit_df = df[df["split"] == "unsplit"].copy()
    if not unsplit_df.empty:
        train_df = pd.concat([train_df, unsplit_df], ignore_index=True)
    if train_df.empty:
        train_df, temp_df = stratified_split(df, test_size=VAL_FRACTION + TEST_FRACTION)
        val_df, test_df = stratified_split(temp_df, test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION))
    if test_df.empty:
        train_df, test_df = stratified_split(train_df, test_size=TEST_FRACTION)
    if val_df.empty:
        val_size_from_remaining = VAL_FRACTION / (TRAIN_FRACTION + VAL_FRACTION)
        train_df, val_df = stratified_split(train_df, test_size=val_size_from_remaining)
else:
    train_df, temp_df = stratified_split(df, test_size=VAL_FRACTION + TEST_FRACTION)
    val_df, test_df = stratified_split(temp_df, test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION))

split_frames = {"train": train_df, "validation": val_df, "test": test_df}
for split_name, split_df in split_frames.items():
    print(f"{split_name}: {len(split_df)} images")

train_counts = train_df["label_index"].value_counts().to_dict()
class_weights = {
    idx: len(train_df) / (num_classes * train_counts.get(idx, 1))
    for idx in range(num_classes)
}
print("Class weights:", {class_names[idx]: round(weight, 3) for idx, weight in class_weights.items()})

dataset_description = pd.DataFrame(
    [
        {
            "dataset_name": "Fresh and Stale Classification",
            "dataset_url": DATASET_URL,
            "dataset_root": str(DATA_ROOT),
            "number_of_classes": num_classes,
            "class_names": ", ".join(class_names),
            "total_images": len(df),
            "train_images": len(train_df),
            "validation_images": len(val_df),
            "test_images": len(test_df),
            "split_ratio": f"{TRAIN_FRACTION:.0%}/{VAL_FRACTION:.0%}/{TEST_FRACTION:.0%}",
            "image_size_used": f"{IMG_SIZE}x{IMG_SIZE}",
        }
    ]
)
save_csv(dataset_description, "dataset_description.csv")

class_distribution = (
    pd.concat(
        [
            train_df.assign(final_split="train"),
            val_df.assign(final_split="validation"),
            test_df.assign(final_split="test"),
        ],
        ignore_index=True,
    )
    .groupby(["final_split", "label"])
    .size()
    .reset_index(name="image_count")
)
save_csv(class_distribution, "class_distribution.csv")

fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * num_classes)))
sns.barplot(data=class_distribution, y="label", x="image_count", hue="final_split", ax=ax)
ax.set_title("Class Distribution by Split")
ax.set_xlabel("Number of Images")
ax.set_ylabel("Class")
save_pdf(fig, "class_distribution.pdf")

display(dataset_description)
display(class_distribution.head(20))


train: 20994 images
validation: 2625 images
test: 6738 images
Class weights: {'freshapples': 0.443, 'freshbanana': 0.435, 'freshbittergroud': 3.279, 'freshcapsicum': 1.084, 'freshcucumber': 2.164, 'freshokra': 1.692, 'freshoranges': 0.732, 'freshpatato': 954.273, 'freshpotato': 2.005, 'freshtamto': 954.273, 'freshtomato': 0.578, 'rottenapples': 0.331, 'rottenbanana': 0.366, 'rottenbittergroud': 3.01, 'rottencapsicum': 1.191, 'rottencucumber': 2.552, 'rottenokra': 3.181, 'rottenoranges': 0.673, 'rottenpatato': 954.273, 'rottenpotato': 1.338, 'rottentamto': 954.273, 'rottentomato': 0.588}
Saved table: /kaggle/working/outputs/tables/dataset_description.csv
Saved table: /kaggle/working/outputs/tables/class_distribution.csv
Saved figure: /kaggle/working/outputs/figures/class_distribution.pdf


,dataset_name,dataset_url,dataset_root,number_of_classes,class_names,total_images,train_images,validation_images,test_images,split_ratio,image_size_used
0,Fresh and Stale Classification,https://www.kaggle.com/datasets/swoyam2609/fre...,/kaggle/input/datasets,22,"freshapples, freshbanana, freshbittergroud, fr...",30357,20994,2625,6738,80%/10%/10%,224x224


,final_split,label,image_count
0,test,freshapples,791
1,test,freshbanana,892
2,test,freshcucumber,279
3,test,freshokra,370
4,test,freshoranges,388
5,test,freshpatato,270
6,test,freshtamto,255
7,test,rottenapples,988
8,test,rottenbanana,900
9,test,rottencucumber,255


## Figure 1: Dataset Samples

In [5]:
sample_rows = []
for label in class_names:
    candidates = train_df[train_df["label"] == label]
    if candidates.empty:
        candidates = df[df["label"] == label]
    sample_rows.extend(candidates.sample(min(4, len(candidates)), random_state=SEED).to_dict("records"))

cols = min(4, max(1, len(sample_rows)))
rows = int(np.ceil(len(sample_rows) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = np.atleast_1d(axes).ravel()

for ax, row in zip(axes, sample_rows):
    ax.imshow(read_resized_image(row["path"]).astype(np.uint8))
    ax.set_title(row["label"])
    ax.axis("off")
for ax in axes[len(sample_rows):]:
    ax.axis("off")

fig.suptitle("Figure 1: Dataset Samples", y=1.02)
save_pdf(fig, "figure_1_dataset_samples.pdf")

Saved figure: /kaggle/working/outputs/figures/figure_1_dataset_samples.pdf


PosixPath('/kaggle/working/outputs/figures/figure_1_dataset_samples.pdf')

## Figure 2: Model Design Workflow

In [6]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")

boxes = [
    ("Input Image", 0.08),
    ("Custom CNN\nMobileNetV2\nResNet50", 0.30),
    ("Prediction", 0.52),
    ("Evaluation\nMetrics + Confusion Matrix", 0.72),
    ("Grad-CAM + SHAP\nExplanation", 0.91),
]

for text, x in boxes:
    ax.text(
        x,
        0.5,
        text,
        ha="center",
        va="center",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.45", facecolor="#f2f2f2", edgecolor="#555555"),
    )

for (_, x1), (_, x2) in zip(boxes[:-1], boxes[1:]):
    ax.annotate(
        "",
        xy=(x2 - 0.09, 0.5),
        xytext=(x1 + 0.09, 0.5),
        arrowprops=dict(arrowstyle="->", lw=1.8, color="#333333"),
    )

ax.set_title("Figure 2: Model Design Workflow")
save_pdf(fig, "figure_2_model_design_workflow.pdf")

Saved figure: /kaggle/working/outputs/figures/figure_2_model_design_workflow.pdf


PosixPath('/kaggle/working/outputs/figures/figure_2_model_design_workflow.pdf')

## TensorFlow Datasets

In [7]:
train_ds = make_dataset(train_df, shuffle=True, class_weights=class_weights)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

print("Classes:")
for idx, label in enumerate(class_names):
    print(f"{idx}: {label}")


I0000 00:00:1780837468.849887      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Classes:
0: freshapples
1: freshbanana
2: freshbittergroud
3: freshcapsicum
4: freshcucumber
5: freshokra
6: freshoranges
7: freshpatato
8: freshpotato
9: freshtamto
10: freshtomato
11: rottenapples
12: rottenbanana
13: rottenbittergroud
14: rottencapsicum
15: rottencucumber
16: rottenokra
17: rottenoranges
18: rottenpatato
19: rottenpotato
20: rottentamto
21: rottentomato


## Model Definitions

In [9]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.10, seed=SEED),
        tf.keras.layers.RandomZoom(0.12, seed=SEED),
        tf.keras.layers.RandomContrast(0.15, seed=SEED),
    ],
    name="data_augmentation",
)


def compile_model(model, learning_rate=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model


def build_custom_cnn():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
    x = data_augmentation(inputs)
    x = tf.keras.layers.Rescaling(1.0 / 255.0)(x)

    for filters in [32, 64, 128, 256]:
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(384, 3, padding="same", activation="relu", name="custom_last_conv")(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.40)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)
    model = tf.keras.Model(inputs, outputs, name="Custom_CNN")
    return compile_model(model, learning_rate=1e-3), "custom_last_conv", None


def build_transfer_model(app_fn, preprocess_fn, model_name, last_conv_layer, dropout_rate=0.30):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
    x = data_augmentation(inputs)
    x = preprocess_fn(x)

    try:
        base = app_fn(include_top=False, weights="imagenet", input_tensor=x)
        imagenet_loaded = True
    except Exception as exc:
        print(f"Could not load ImageNet weights for {model_name}: {type(exc).__name__}: {exc}")
        print(f"Using random weights for {model_name}; accuracy may be lower without internet/cache access.")
        base = app_fn(include_top=False, weights=None, input_tensor=x)
        imagenet_loaded = False

    base.trainable = not imagenet_loaded
    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)
    model = tf.keras.Model(inputs, outputs, name=model_name)
    learning_rate = 5e-4 if imagenet_loaded else 1e-4
    fine_tune_base = base if imagenet_loaded else None
    return compile_model(model, learning_rate=learning_rate), last_conv_layer, fine_tune_base


def build_mobilenetv2():
    return build_transfer_model(
        tf.keras.applications.MobileNetV2,
        tf.keras.applications.mobilenet_v2.preprocess_input,
        "MobileNetV2_Transfer",
        "out_relu",
        dropout_rate=0.25,
    )


def build_resnet50():
    return build_transfer_model(
        tf.keras.applications.ResNet50,
        tf.keras.applications.resnet50.preprocess_input,
        "ResNet50_Transfer",
        "conv5_block3_out",
        dropout_rate=0.35,
    )


def make_callbacks(model_name, phase):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=4,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


def append_history(history, extra_history):
    if extra_history is None:
        return history
    for key, values in extra_history.history.items():
        history.history.setdefault(key, []).extend(values)
    return history


def fine_tune_model(model, base_model, model_name):
    if base_model is None or FINE_TUNE_EPOCHS <= 0:
        return None

    base_model.trainable = True
    freeze_until = max(0, len(base_model.layers) - FINE_TUNE_LAST_LAYERS)
    for layer in base_model.layers[:freeze_until]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    compile_model(model, learning_rate=1e-5)
    print(f"Fine-tuning last {FINE_TUNE_LAST_LAYERS} layers for {model_name}")
    return model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=make_callbacks(model_name, "fine_tune"),
        verbose=1,
    )


model_builders = {
    "Custom CNN": build_custom_cnn,
    "MobileNetV2": build_mobilenetv2,
    "ResNet50": build_resnet50,
}


## Part A and B: Train, Evaluate, and Save Tables

In [10]:
histories = {}
trained_models = {}
last_conv_layers = {}
metrics_rows = []


''
prediction_tables = {}

y_true = test_df["
label_index"].astype(int).values

for model_name, builder in model_builders.items():
    print(f"\n===== Training {model_name} =====")
    tf.keras.backend.clear_session()
    model, last_conv_layer, fine_tune_base = builder()
    start_time = time.perf_counter()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=make_callbacks(model_name, "initial"),
        verbose=1,
    )
    fine_tune_history = fine_tune_model(model, fine_tune_base, model_name)
    history = append_history(history, fine_tune_history)
    training_time = time.perf_counter() - start_time

    probabilities = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)
    confidence = np.max(probabilities, axis=1)

    metrics_rows.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "training_time_seconds": training_time,
            "total_parameters": model.count_params(),
            "trainable_parameters": int(np.sum([np.prod(v.shape) for v in model.trainable_weights])),
            "last_conv_layer": last_conv_layer,
        }
    )

    prediction_df = test_df[["path", "label", "label_index"]].copy()
    prediction_df["predicted_index"] = y_pred
    prediction_df["predicted_label"] = [class_names[i] for i in y_pred]
    prediction_df["confidence"] = confidence
    prediction_df["is_correct"] = prediction_df["label_index"] == prediction_df["predicted_index"]
    prediction_tables[model_name] = prediction_df
    save_csv(prediction_df, f"predictions_{make_slug(model_name)}.csv")

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(num_classes)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose().reset_index().rename(columns={"index": "class_or_average"})
    save_csv(report_df, f"classification_report_{make_slug(model_name)}.csv")

    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    history_df.insert(0, "model", model_name)
    save_csv(history_df, f"training_history_{make_slug(model_name)}.csv")

    model.save(MODEL_DIR / f"{make_slug(model_name)}.keras")
    histories[model_name] = history
    trained_models[model_name] = model
    last_conv_layers[model_name] = last_conv_layer

evaluation_results = pd.DataFrame(metrics_rows).sort_values("f1_weighted", ascending=False)
save_csv(evaluation_results, "evaluation_results.csv")

architecture_table = evaluation_results[
    ["model", "total_parameters", "trainable_parameters", "last_conv_layer"]
].copy()
architecture_table["input_size"] = f"{IMG_SIZE}x{IMG_SIZE}x3"
architecture_table["training_strategy"] = [
    "Convolutional neural network trained from scratch" if model == "Custom CNN"
    else "ImageNet backbone, classifier head training, then fine-tuning"
    for model in architecture_table["model"]
]
save_csv(architecture_table, "model_architecture_table.csv")

display(evaluation_results)
display(architecture_table)



===== Training Custom CNN =====
Epoch 1/20


I0000 00:00:1780837501.954329     161 cuda_dnn.cc:529] Loaded cuDNN version 91002


657/657 ━━━━━━━━━━━━━━━━━━━━ 125s 171ms/step - accuracy: 0.5775 - loss: 1.1284 - val_accuracy: 0.3219 - val_loss: 3.6571 - learning_rate: 0.0010
Epoch 2/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 109s 165ms/step - accuracy: 0.7193 - loss: 0.7211 - val_accuracy: 0.3330 - val_loss: 2.6908 - learning_rate: 0.0010
Epoch 3/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 108s 165ms/step - accuracy: 0.7795 - loss: 0.5674 - val_accuracy: 0.5330 - val_loss: 3.3391 - learning_rate: 0.0010
Epoch 4/20
656/657 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 0.7895 - loss: 0.5549
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
657/657 ━━━━━━━━━━━━━━━━━━━━ 108s 165ms/step - accuracy: 0.8082 - loss: 0.5072 - val_accuracy: 0.3790 - val_loss: 4.1194 - learning_rate: 0.0010
Epoch 5/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 108s 165ms/step - accuracy: 0.8636 - loss: 0.3645 - val_accuracy: 0.5931 - val_loss: 2.5093 - learning_rate: 3.0000e-04
Epoch 6/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 108s 165ms/step - accuracy: 0.8883

/tmp/ipykernel_58/642587591.py:47: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = app_fn(include_top=False, weights="imagenet", input_tensor=x)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 39s 51ms/step - accuracy: 0.8227 - loss: 0.5648 - val_accuracy: 0.9364 - val_loss: 0.2211 - learning_rate: 5.0000e-04
Epoch 2/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 32s 48ms/step - accuracy: 0.9427 - loss: 0.2181 - val_accuracy: 0.9585 - val_loss: 0.1540 - learning_rate: 5.0000e-04
Epoch 3/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 31s 48ms/step - accuracy: 0.9552 - loss: 0.1726 - val_accuracy: 0.9600 - val_loss: 0.1293 - learning_rate: 5.0000e-04
Epoch 4/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 32s 48ms/step - accuracy: 0.9598 - loss: 0.1493 - val_accuracy: 0.9650 - val_loss: 0.1188 - learning_rate: 5.0000e-04
Epoch 5/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 31s 48ms/step - accuracy: 0.9665 - loss: 0.1290 - val_accuracy: 0.9676 - val_loss: 0.1022 - learning_rate: 5.0000e-04
Epoch 6/20
657/657 ━━━━━━━━━━━━━━━━━━━━ 32s 48ms/step - accuracy: 0.9699 - loss: 0.1209 - val_accuracy: 0.9688 - val_loss: 0.0952 - learning_rate: 5.0000e-04
Epo

,model,accuracy,precision_weighted,recall_weighted,f1_weighted,training_time_seconds,total_parameters,trainable_parameters,last_conv_layer
2,ResNet50,0.810775,0.810695,0.810775,0.810702,1653.223198,23632790,15851798,conv5_block3_out
1,MobileNetV2,0.808103,0.808446,0.808103,0.808246,679.258689,2286166,1691542,out_relu
0,Custom CNN,0.777976,0.781144,0.777976,0.779112,2180.331670,2069686,2067766,custom_last_conv


,model,total_parameters,trainable_parameters,last_conv_layer,input_size,training_strategy
2,ResNet50,23632790,15851798,conv5_block3_out,224x224x3,"ImageNet backbone, classifier head training, t..."
1,MobileNetV2,2286166,1691542,out_relu,224x224x3,"ImageNet backbone, classifier head training, t..."
0,Custom CNN,2069686,2067766,custom_last_conv,224x224x3,Convolutional neural network trained from scratch


## Figure 3: Accuracy and Loss Curves

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name, history in histories.items():
    history_df = pd.DataFrame(history.history)
    epochs = np.arange(1, len(history_df) + 1)
    axes[0].plot(epochs, history_df["accuracy"], marker="o", label=f"{model_name} Train")
    axes[0].plot(epochs, history_df["val_accuracy"], marker="s", linestyle="--", label=f"{model_name} Val")
    axes[1].plot(epochs, history_df["loss"], marker="o", label=f"{model_name} Train")
    axes[1].plot(epochs, history_df["val_loss"], marker="s", linestyle="--", label=f"{model_name} Val")

axes[0].set_title("Accuracy Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend(fontsize=8)

axes[1].set_title("Loss Curves")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend(fontsize=8)

fig.suptitle("Figure 3: Accuracy and Loss Curves")
save_pdf(fig, "figure_3_accuracy_loss_curves.pdf")

Saved figure: /kaggle/working/outputs/figures/figure_3_accuracy_loss_curves.pdf


PosixPath('/kaggle/working/outputs/figures/figure_3_accuracy_loss_curves.pdf')

## Figure 4: Confusion Matrix Comparison

In [12]:
fig, axes = plt.subplots(1, len(trained_models), figsize=(6 * len(trained_models), 5))
axes = np.atleast_1d(axes)

for ax, (model_name, prediction_df) in zip(axes, prediction_tables.items()):
    cm = confusion_matrix(
        prediction_df["label_index"],
        prediction_df["predicted_index"],
        labels=list(range(num_classes)),
    )
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    save_csv(cm_df.reset_index().rename(columns={"index": "true_label"}), f"confusion_matrix_{make_slug(model_name)}.csv")

    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(model_name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig.suptitle("Figure 4: Confusion Matrix Comparison", y=1.04)
save_pdf(fig, "figure_4_confusion_matrix_comparison.pdf")


Saved table: /kaggle/working/outputs/tables/confusion_matrix_custom_cnn.csv
Saved table: /kaggle/working/outputs/tables/confusion_matrix_mobilenetv2.csv
Saved table: /kaggle/working/outputs/tables/confusion_matrix_resnet50.csv
Saved figure: /kaggle/working/outputs/figures/figure_4_confusion_matrix_comparison.pdf


PosixPath('/kaggle/working/outputs/figures/figure_4_confusion_matrix_comparison.pdf')

## Part C: Grad-CAM Explainability

In [13]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output],
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap(img_array, heatmap, alpha=0.45):
    img = img_array[0].astype(np.uint8)
    cmap = plt.colormaps["jet"]
    heatmap_rgb = np.uint8(255 * cmap(heatmap)[..., :3])
    heatmap_img = Image.fromarray(heatmap_rgb).resize((IMG_SIZE, IMG_SIZE))
    heatmap_img = np.asarray(heatmap_img)
    return np.uint8(heatmap_img * alpha + img * (1 - alpha))


def plot_gradcam_comparison(row, filename, figure_title):
    img = read_resized_image(row["path"])
    img_batch = np.expand_dims(img, axis=0)
    fig, axes = plt.subplots(1, len(trained_models) + 1, figsize=(4 * (len(trained_models) + 1), 4))
    axes[0].imshow(img.astype(np.uint8))
    axes[0].set_title(f"Original\nTrue: {row['label']}")
    axes[0].axis("off")

    table_rows = []
    for ax, (model_name, model) in zip(axes[1:], trained_models.items()):
        probs = model.predict(img_batch, verbose=0)[0]
        pred_idx = int(np.argmax(probs))
        pred_label = class_names[pred_idx]
        heatmap = get_gradcam_heatmap(model, img_batch, last_conv_layers[model_name], pred_idx)
        overlay = overlay_heatmap(img_batch, heatmap)
        ax.imshow(overlay)
        ax.set_title(f"{model_name}\nPred: {pred_label}\nConf: {probs[pred_idx]:.2f}")
        ax.axis("off")
        table_rows.append(
            {
                "figure": filename,
                "model": model_name,
                "image_path": row["path"],
                "true_label": row["label"],
                "predicted_label": pred_label,
                "confidence": probs[pred_idx],
                "is_correct": pred_label == row["label"],
            }
        )

    fig.suptitle(figure_title, y=1.05)
    save_pdf(fig, filename)
    return table_rows


best_model_name = evaluation_results.iloc[0]["model"]
best_predictions = prediction_tables[best_model_name]

correct_candidates = best_predictions[best_predictions["is_correct"]].copy()
if correct_candidates.empty:
    correct_row = best_predictions.sort_values("confidence", ascending=False).iloc[0].to_dict()
else:
    correct_row = correct_candidates.sort_values("confidence", ascending=False).iloc[0].to_dict()

incorrect_candidates = best_predictions[~best_predictions["is_correct"]].copy()
if incorrect_candidates.empty:
    incorrect_row = best_predictions.sort_values("confidence", ascending=True).iloc[0].to_dict()
    incorrect_title = "Figure 6: Grad-CAM Low-Confidence Example"
else:
    incorrect_row = incorrect_candidates.sort_values("confidence", ascending=False).iloc[0].to_dict()
    incorrect_title = "Figure 6: Grad-CAM Misclassification Examples"

gradcam_rows = []
gradcam_rows.extend(
    plot_gradcam_comparison(
        correct_row,
        "figure_5_gradcam_correct_prediction_examples.pdf",
        "Figure 5: Grad-CAM Correct Prediction Examples",
    )
)
gradcam_rows.extend(
    plot_gradcam_comparison(
        incorrect_row,
        "figure_6_gradcam_misclassification_examples.pdf",
        incorrect_title,
    )
)
save_csv(pd.DataFrame(gradcam_rows), "gradcam_examples.csv")

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['image']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


Saved figure: /kaggle/working/outputs/figures/figure_5_gradcam_correct_prediction_examples.pdf


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['image']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


Saved figure: /kaggle/working/outputs/figures/figure_6_gradcam_misclassification_examples.pdf
Saved table: /kaggle/working/outputs/tables/gradcam_examples.csv


PosixPath('/kaggle/working/outputs/tables/gradcam_examples.csv')

## Figure 7: SHAP Explanation Examples

In [14]:
def save_shap_for_model(model_name, model, sample_images):
    model_slug = make_slug(model_name)
    filename = f"figure_7_shap_explanation_{model_slug}.pdf"

    if shap is None:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.axis("off")
        ax.text(0.5, 0.5, "SHAP is not installed, so this explanation was skipped.", ha="center", va="center")
        ax.set_title(f"Figure 7: SHAP Explanation - {model_name}")
        save_pdf(fig, filename)
        return {
            "model": model_name,
            "figure": filename,
            "status": "skipped",
            "notes": "Install shap to generate SHAP image explanations.",
        }

    def predict_fn(images):
        images = np.asarray(images, dtype=np.float32)
        return model.predict(images, verbose=0)

    try:
        masker = shap.maskers.Image("blur(32,32)", sample_images[0].shape)
        explainer = shap.Explainer(predict_fn, masker, output_names=class_names)
        shap_values = explainer(
            sample_images,
            max_evals=SHAP_MAX_EVALS,
            batch_size=SHAP_BATCH_SIZE,
            outputs=shap.Explanation.argsort.flip[:1],
        )
        shap.image_plot(shap_values, show=False)
        fig = plt.gcf()
        fig.suptitle(f"Figure 7: SHAP Explanation - {model_name}", y=1.02)
        save_pdf(fig, filename)
        return {
            "model": model_name,
            "figure": filename,
            "status": "saved",
            "notes": f"Generated with max_evals={SHAP_MAX_EVALS}",
        }
    except Exception as exc:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.axis("off")
        ax.text(
            0.5,
            0.5,
            f"SHAP generation failed for {model_name}\n{type(exc).__name__}: {exc}",
            ha="center",
            va="center",
            wrap=True,
        )
        ax.set_title(f"Figure 7: SHAP Explanation - {model_name}")
        save_pdf(fig, filename)
        return {
            "model": model_name,
            "figure": filename,
            "status": "error_placeholder_saved",
            "notes": f"{type(exc).__name__}: {exc}",
        }


shap_source = correct_candidates if "correct_candidates" in globals() and not correct_candidates.empty else best_predictions
shap_rows_df = shap_source.head(SHAP_IMAGE_COUNT)
shap_images = np.stack([read_resized_image(path) for path in shap_rows_df["path"].values], axis=0)

shap_status_rows = []
for model_name, model in trained_models.items():
    shap_status_rows.append(save_shap_for_model(model_name, model, shap_images))

shap_status = pd.DataFrame(shap_status_rows)
save_csv(shap_status, "shap_examples.csv")
display(shap_status)


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [1.0..255.0].


Saved figure: /kaggle/working/outputs/figures/figure_7_shap_explanation_custom_cnn.pdf


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [1.0..255.0].


Saved figure: /kaggle/working/outputs/figures/figure_7_shap_explanation_mobilenetv2.pdf


  0%|          | 0/298 [00:00<?, ?it/s]

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [1.0..255.0].


Saved figure: /kaggle/working/outputs/figures/figure_7_shap_explanation_resnet50.pdf
Saved table: /kaggle/working/outputs/tables/shap_examples.csv


,model,figure,status,notes
0,Custom CNN,figure_7_shap_explanation_custom_cnn.pdf,saved,Generated with max_evals=300
1,MobileNetV2,figure_7_shap_explanation_mobilenetv2.pdf,saved,Generated with max_evals=300
2,ResNet50,figure_7_shap_explanation_resnet50.pdf,saved,Generated with max_evals=300


## Final Comparison Table and Discussion Inputs

In [15]:
comparison_table = evaluation_results[
    [
        "model",
        "accuracy",
        "f1_weighted",
        "training_time_seconds",
        "total_parameters",
    ]
].copy()

comparison_table["gradcam_quality"] = "Review Figure 5 and Figure 6"
comparison_table["shap_interpretability"] = "Review Figure 7"
best_model = comparison_table.sort_values("f1_weighted", ascending=False).iloc[0]["model"]
fastest_model = comparison_table.sort_values("training_time_seconds", ascending=True).iloc[0]["model"]

comparison_table["overall_finding"] = comparison_table["model"].apply(
    lambda name: (
        "Best weighted F1-score on the test set"
        if name == best_model
        else "Fastest training time among compared models"
        if name == fastest_model
        else "Useful baseline for comparison"
    )
)

save_csv(comparison_table, "final_comparison_table.csv")

research_question_summary = pd.DataFrame(
    [
        {
            "research_question": "RQ1",
            "answer_basis": "Custom CNN row in evaluation_results.csv and its confusion matrix.",
        },
        {
            "research_question": "RQ2",
            "answer_basis": "Compare Custom CNN against MobileNetV2 and ResNet50 in final_comparison_table.csv.",
        },
        {
            "research_question": "RQ3",
            "answer_basis": "Compare weighted F1-score, training_time_seconds, and total_parameters.",
        },
        {
            "research_question": "RQ4",
            "answer_basis": "Inspect Grad-CAM correct-prediction examples in Figure 5.",
        },
        {
            "research_question": "RQ5",
            "answer_basis": "Compare attention regions across model columns in Figures 5 and 6.",
        },
        {
            "research_question": "RQ6",
            "answer_basis": "Inspect positive and negative evidence in Figure 7 SHAP explanations.",
        },
        {
            "research_question": "RQ7",
            "answer_basis": "Use Figure 6 and gradcam_examples.csv to diagnose misclassified or low-confidence images.",
        },
    ]
)
save_csv(research_question_summary, "research_question_summary.csv")

display(comparison_table)
display(research_question_summary)

Saved table: /kaggle/working/outputs/tables/final_comparison_table.csv
Saved table: /kaggle/working/outputs/tables/research_question_summary.csv


,model,accuracy,f1_weighted,training_time_seconds,total_parameters,gradcam_quality,shap_interpretability,overall_finding
2,ResNet50,0.810775,0.810702,1653.223198,23632790,Review Figure 5 and Figure 6,Review Figure 7,Best weighted F1-score on the test set
1,MobileNetV2,0.808103,0.808246,679.258689,2286166,Review Figure 5 and Figure 6,Review Figure 7,Fastest training time among compared models
0,Custom CNN,0.777976,0.779112,2180.331670,2069686,Review Figure 5 and Figure 6,Review Figure 7,Useful baseline for comparison


,research_question,answer_basis
0,RQ1,Custom CNN row in evaluation_results.csv and i...
1,RQ2,Compare Custom CNN against MobileNetV2 and Res...
2,RQ3,"Compare weighted F1-score, training_time_secon..."
3,RQ4,Inspect Grad-CAM correct-prediction examples i...
4,RQ5,Compare attention regions across model columns...
5,RQ6,Inspect positive and negative evidence in Figu...
6,RQ7,Use Figure 6 and gradcam_examples.csv to diagn...


## Output Checklist

In [16]:
output_files = []
for folder in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    output_files.extend(sorted(folder.glob("*")))

output_manifest = pd.DataFrame(
    {
        "path": [str(path) for path in output_files],
        "file_type": [path.suffix.lower() for path in output_files],
        "size_bytes": [path.stat().st_size for path in output_files],
    }
)
save_csv(output_manifest, "output_manifest.csv")
display(output_manifest)

print("\nFigures saved as PDF:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(path)

print("\nTables saved as CSV:")
for path in sorted(TABLE_DIR.glob("*.csv")):
    print(path)

Saved table: /kaggle/working/outputs/tables/output_manifest.csv


,path,file_type,size_bytes
0,/kaggle/working/outputs/figures/class_distribu...,.pdf,15201
1,/kaggle/working/outputs/figures/figure_1_datas...,.pdf,3996387
2,/kaggle/working/outputs/figures/figure_2_model...,.pdf,15526
3,/kaggle/working/outputs/figures/figure_3_accur...,.pdf,25202
4,/kaggle/working/outputs/figures/figure_4_confu...,.pdf,46517
5,/kaggle/working/outputs/figures/figure_5_gradc...,.pdf,532125
6,/kaggle/working/outputs/figures/figure_6_gradc...,.pdf,254184
7,/kaggle/working/outputs/figures/figure_7_shap_...,.pdf,50297
8,/kaggle/working/outputs/figures/figure_7_shap_...,.pdf,51695
9,/kaggle/working/outputs/figures/figure_7_shap_...,.pdf,50312



Figures saved as PDF:
/kaggle/working/outputs/figures/class_distribution.pdf
/kaggle/working/outputs/figures/figure_1_dataset_samples.pdf
/kaggle/working/outputs/figures/figure_2_model_design_workflow.pdf
/kaggle/working/outputs/figures/figure_3_accuracy_loss_curves.pdf
/kaggle/working/outputs/figures/figure_4_confusion_matrix_comparison.pdf
/kaggle/working/outputs/figures/figure_5_gradcam_correct_prediction_examples.pdf
/kaggle/working/outputs/figures/figure_6_gradcam_misclassification_examples.pdf
/kaggle/working/outputs/figures/figure_7_shap_explanation_custom_cnn.pdf
/kaggle/working/outputs/figures/figure_7_shap_explanation_mobilenetv2.pdf
/kaggle/working/outputs/figures/figure_7_shap_explanation_resnet50.pdf

Tables saved as CSV:
/kaggle/working/outputs/tables/class_distribution.csv
/kaggle/working/outputs/tables/classification_report_custom_cnn.csv
/kaggle/working/outputs/tables/classification_report_mobilenetv2.csv
/kaggle/working/outputs/tables/classification_report_resnet50.c

In [17]:
import zipfile
import os
from pathlib import Path
from IPython.display import FileLink

# ── Paths from your notebook ──────────────────────────────────────────────────
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
OUTPUT_DIR = KAGGLE_WORKING_ROOT / "outputs"
FIG_DIR    = OUTPUT_DIR / "figures"
TABLE_DIR  = OUTPUT_DIR / "tables"
MODEL_DIR  = OUTPUT_DIR / "models"

output_filename = KAGGLE_WORKING_ROOT / "cnn_project_export.zip"

print("Archiving CNN project outputs...")

with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:

    # 1. All PDF figures
    for pdf in sorted(FIG_DIR.glob("*.pdf")):
        zipf.write(pdf, f"figures/{pdf.name}")

    # 2. All CSV tables
    for csv in sorted(TABLE_DIR.glob("*.csv")):
        zipf.write(csv, f"tables/{csv.name}")

    # 3. All saved Keras models
    for model_file in sorted(MODEL_DIR.glob("*.keras")):
        zipf.write(model_file, f"models/{model_file.name}")

    # 4. The notebook itself
    nb_path = KAGGLE_WORKING_ROOT / "cnn-updated.ipynb"
    if nb_path.exists():
        zipf.write(nb_path, "cnn-updated.ipynb")

# Summary
with zipfile.ZipFile(output_filename, 'r') as zipf:
    names = zipf.namelist()

print(f"✓ Archive created: {output_filename}")
print(f"  Total files: {len(names)}")
print(f"  PDFs  (figures): {sum(1 for n in names if n.startswith('figures/'))}")
print(f"  CSVs  (tables):  {sum(1 for n in names if n.startswith('tables/'))}")
print(f"  Keras (models):  {sum(1 for n in names if n.startswith('models/'))}")

display(FileLink(str(output_filename.relative_to(KAGGLE_WORKING_ROOT))))

Archiving CNN project outputs...
✓ Archive created: /kaggle/working/cnn_project_export.zip
  Total files: 34
  PDFs  (figures): 10
  CSVs  (tables):  21
  Keras (models):  3


/kaggle/working/cnn_project_export.zip